# 🚀 HOLO — Google Play AAB Oluşturucu (Colab)

Google Play'e yüklenecek **imzalı AAB** dosyasını derler. Test araçları kapalıdır, bölüm seçimi yoktur, Play Faturalandırma açıktır.

1. Bilgisayarında `python tools\make_apk_zip.py --release` çalıştır → `holo_release_source.zip`.
2. Hücreleri sırayla çalıştır. 2. adımda o zip'i yükle.
3. 3. adımda **imza anahtarı** (keystore) oluşturulur: şifreyi sen yazarsın (kimseyle paylaşma). **İndirilen `holo-upload.jks` dosyasını ve şifresini güvenli yerde sakla**; sonraki her güncelleme aynı anahtarla imzalanmalı.
4. 4. adım AAB'yi derleyip indirir (ilk derleme 25-40 dk). Play Console → Test → Dahili test → yeni sürüm oluştur → AAB'yi yükle.


### 1. Adım: Linux paketleri ve Buildozer

In [ ]:
!sudo apt update -qq
!sudo apt install -y -qq git zip unzip openjdk-17-jdk python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev
!pip install --upgrade -q Cython==0.29.36 buildozer virtualenv
print("✅ Hazır")

### 2. Adım: `holo_release_source.zip` yükle

In [ ]:
from google.colab import files
import os, shutil, zipfile
up = files.upload()
name = list(up.keys())[0]
target = "/content/holo_project"
shutil.rmtree(target, ignore_errors=True); os.makedirs(target)
zipfile.ZipFile(name).extractall(target)
os.chdir(target)
print(open("config.json").read())
print("✅ Proje hazır (config'te test_tools yok olmalı, billing=android olmalı)")

### 3. Adım: İmza anahtarı (keystore)

Anahtar ve şifresi otomatik üretilir ve iki dosya olarak iner (`holo-upload.jks` + `holo-keystore-bilgi.txt`). **İkisini de sakla**: sonraki güncellemeler aynı anahtarla imzalanır. Zaten varsa iki dosyayı da 2. adımdaki zip'in yanına yükle.

In [ ]:
import secrets, subprocess, os, json
from google.colab import files
ks = "/content/holo_project/holo-upload.jks"
alias = "holo"
info = "/content/holo_project/holo-keystore-bilgi.txt"
if os.path.exists(ks) and os.path.exists(info):
    pw = json.load(open(info))["password"]                     # keystore uploaded together with its info file
else:
    pw = secrets.token_urlsafe(18)                               # random password: nobody has to type or share one
    subprocess.run(["keytool","-genkeypair","-v","-keystore",ks,"-alias",alias,"-keyalg","RSA","-keysize","2048","-validity","10000",
                    "-storepass",pw,"-keypass",pw,"-dname","CN=HOLO, OU=Games, O=HOLO, C=TR"], check=True)
    json.dump({"keystore": "holo-upload.jks", "alias": alias, "password": pw}, open(info, "w"))
    print("Anahtar olusturuldu. holo-upload.jks ve holo-keystore-bilgi.txt indiriliyor: IKISINI DE GUVENLI YERDE SAKLA (yedekle).")
    files.download(ks); files.download(info)
os.environ["P4A_RELEASE_KEYSTORE"] = ks
os.environ["P4A_RELEASE_KEYALIAS"] = alias
os.environ["P4A_RELEASE_KEYSTORE_PASSWD"] = pw
os.environ["P4A_RELEASE_KEYALIAS_PASSWD"] = pw

### 4. Adım: AAB derle ve indir

In [ ]:
import glob
from google.colab import files
!buildozer -v android release
aabs = glob.glob("bin/*.aab")
if aabs:
    print("🎉 AAB hazır:", aabs[0]); files.download(aabs[0])
else:
    print("❌ AAB bulunamadı: yukarıdaki logu incele / bana gönder")